# S01b — REHAB24-6 Ex1 + Ex2 shoulder features (cells for `benchmark_3d_world.ipynb` on Colab)

REHAB24-6 Ex1 = arm abduction, Ex2 = arm VW. Both cameras (cam17 and cam18, ~90° apart) film the same
rep, so this gives **front and side views** of the same movement.

**Order**
1. Push the repo with `shoulder_features.py` to GitHub.
2. Colab: Runtime → Restart, then STEP 1 → STEP 5 of `benchmark_3d_world.ipynb`
   (`EXERCISES = [1, 2, 3]` in STEP 2; the landmark cache already holds Ex1 and Ex2, so this is quick).
3. Paste **cell SH1** after STEP 5 and run it: it checks the joint names.
4. Paste **cell SH2** and run it → `RESULTS/feat_rehab24_shoulder.npz`.
5. Download that file into the repo's `data/fit3d/` folder on your PC.

In [ ]:
# ===== CELL SH1 - shoulder, elbow and hip joints in the REHAB24-6 skeleton =====
import importlib
import shoulder_features; importlib.reload(shoulder_features)
from shoulder_features import SH_FEATURE_NAMES, IDX, stream_shoulder_features, elevation

for _need in ('recordings', 'get_3d_landmarks', 'JOINT_NAMES', 'resample_to', 'infer_view', 'CACHE', 'RESULTS'):
    assert _need in dir(), f'{_need} missing - run STEP 1 to STEP 5 first'

def find_joint(*cands):
    for cand in cands:
        for n in JOINT_NAMES:
            if n.split(':', 1)[-1].strip().lower() == cand.lower():
                return n
    return None

RJ = {}
for side, S in (('left', 'Left'), ('right', 'Right')):
    RJ[side] = dict(sh=ARMS_OVERRIDE[side][0], el=ARMS_OVERRIDE[side][1],
                    hip=find_joint(f'{S}UpLeg', f'{S}Hip', f'{S}Thigh', f'{S}_hip', f'{S}Leg'))
print('joint names in the file:', JOINT_NAMES)
print('chosen:', RJ)
assert all(v for d in RJ.values() for v in d.values()), \
    'a hip joint was not found - send the joint-name list printed above'
JI = {s: {k: JOINT_NAMES.index(v) for k, v in d.items()} for s, d in RJ.items()}

In [ ]:
# ===== CELL SH2 - truth, streaming features, gates, save =====
import time
import numpy as np
from features import overlap_rows

SH_EX, MIN_ROM_SH, MIN_COV_SH = (1, 2), 40.0, 50.0
I2D = IDX['angle_2d']

def truth_sh(truth_arr, side):
    J = truth_arr[:, :, :3]
    mid_sh = (J[:, JI['left']['sh']] + J[:, JI['right']['sh']]) / 2
    mid_hip = (J[:, JI['left']['hip']] + J[:, JI['right']['hip']]) / 2
    return elevation(mid_hip, mid_sh, J[:, JI[side]['sh']], J[:, JI[side]['el']])

parts, skipped, t0 = [], {'static': 0, 'occluded': 0, 'align': 0, 'error': 0}, time.time()
recs = [r for r in recordings if r['exercise'] in SH_EX]
for k, rec in enumerate(recs, 1):
    try:
        lm, world, size, fps = get_3d_landmarks(rec)
        truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
        npz_f0 = int(np.load(os.path.join(CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"))['f0'])
        TRUTH_FPS = 120.0 if '120fps' in str(rec.get('truth_key', '')) else 30.0
        for side in ('right', 'left'):
            if abs(TRUTH_FPS - fps) < 1.0:
                gt = truth_sh(truth_arr[npz_f0: npz_f0 + len(world)], side)
            else:
                gt = resample_to((npz_f0 + np.arange(len(world))) / fps,
                                 np.arange(len(truth_arr)) / TRUTH_FPS, truth_sh(truth_arr, side))
            if not np.isfinite(gt).any() or np.nanpercentile(gt, 95) - np.nanpercentile(gt, 5) < MIN_ROM_SH:
                skipped['static'] += 1; continue
            F = stream_shoulder_features(lm, world, side, size, fps, threshold=THRESHOLD)
            if 100.0 * np.isfinite(F[:, I2D]).mean() < MIN_COV_SH:
                skipped['occluded'] += 1; continue
            lag, lag_r = bc.best_lag(F[:, I2D], gt, max_lag=MAX_LAG, filter_signals=True)
            if abs(lag) >= MAX_LAG or lag_r < 0.7:
                skipped['align'] += 1; continue
            Fa, g = overlap_rows(F, gt, lag)
            ok = np.isfinite(g) & np.isfinite(Fa[:, I2D])
            parts.append((Fa[ok], g[ok], bc.person_of(rec['video_key']), f"{rec['rec_id']}_{side}",
                          infer_view(rec), rec['exercise']))
    except Exception as e:
        skipped['error'] += 1
        print('  error', rec['rec_id'], type(e).__name__, e)
    print(f'[{k}/{len(recs)}] {rec["rec_id"]}  {time.time() - t0:.0f}s', flush=True)

print('kept', len(parts), 'arm-recordings; skipped', skipped)
n = [len(p[1]) for p in parts]
out = os.path.join(RESULTS, 'feat_rehab24_shoulder.npz')
np.savez_compressed(
    out,
    X=np.concatenate([p[0] for p in parts]).astype(np.float32),
    y=np.concatenate([p[1] for p in parts]).astype(np.float32),
    feature_names=np.array(SH_FEATURE_NAMES),
    person=np.repeat(['rehab_' + str(p[2]) for p in parts], n),
    recording=np.repeat(['rehab_' + str(p[3]) for p in parts], n),
    view=np.repeat([str(p[4]) for p in parts], n),
    exercise=np.repeat([f'ex{p[5]}' for p in parts], n),
    dataset=np.array(['rehab24'] * sum(n)),
)
raw = [float(np.mean(np.abs(p[0][:, I2D] - p[1]))) for p in parts]
print(f'saved {out}: {sum(n):,} frames, {len(parts)} arm-recordings, {len(set(p[2] for p in parts))} people')
print('raw baseline (median per-recording MAE of the live angle): %.2f deg' % np.median(raw))